# Document Intelligence Playground

This notebook demonstrates the end-to-end document processing pipeline, including advanced parsing, ingestion, and hybrid reranking.

### Pipeline Overview:
1. **Parsing**: Extract structure and content using PaddleOCR-VL (local or remote).
2. **Ingestion**: Securely store document chunks in Qdrant with Merkle integrity verification.
3. **Hybrid Retrieval**: Combine vector and sparse scores (RRF).
4. **Reranking**: Use a Cross-Encoder for high-precision citation-aware results.

In [ ]:
import os
import sys
import json
import asyncio
import nest_asyncio
from pathlib import Path

# Allow nested event loops in Jupyter
nest_asyncio.apply()

# Add 'src' to python path
if "./src" not in sys.path:
    sys.path.insert(0, "./src")

from shared.schemas import Document, PipelineSettings
from document_parser import DocumentParser
from utils.interactive_layout import display_layout_interactive_batch

## 1. Advanced Document Parsing

Configure the parser with the new advanced settings for layout detection and VLM sampling.

In [ ]:
# Option A: Local MLX-VLM Server (assuming running on port 8080)
mlx_settings = PipelineSettings(
    vl_rec_backend="mlx-vlm-server",
    vl_rec_server_url="http://localhost:8080/v1",
    layout_threshold=0.6,       # Stricter layout detection
    temperature=0.0,            # Deterministic VLM output
    max_new_tokens=512          # Support longer text blocks
)

# Option B: Native PaddleOCR-VL (In-process execution)
native_settings = PipelineSettings(
    vl_rec_backend="native",
    use_layout_detection=True,
    layout_nms=True
)

# Initialize parser with MLX settings
parser = DocumentParser(mlx_settings)

In [ ]:
# Parse a single document
sample_path = "data/report.png" # Replace with your file
if os.path.exists(sample_path):
    docs = parser.parse(sample_path)
    print(f"Parsed {len(docs)} pages.")
    
    # Visualize layout
    display_layout_interactive_batch(docs, min_confidence=0.5)
else:
    print(f"File not found: {sample_path}")

### Parallel Batch Parsing
Demonstrate scalable processing using multiple worker processes.

In [ ]:
batch_paths = ["data/report.png", "data/handwritten.jpg"] # Sample paths
existing_paths = [p for p in batch_paths if os.path.exists(p)]

if existing_paths:
    batch_results = parser.parse_batch(existing_paths, max_workers=2)
    for i, res in enumerate(batch_results):
        print(f"Document {i+1}: {len(res)} pages")

## 2. Secure Ingestion with Merkle Integrity

Store the parsed blocks in Qdrant and verify their provenance using Merkle roots.

In [ ]:
from ingestion_pipeline import AsyncMerkleQdrantIngestor

async def run_ingestion(docs_to_ingest):
    ingestor = AsyncMerkleQdrantIngestor(
        qdrant_url="http://localhost:6333",
        redis_host="localhost",
        collection_base_name="secured_docs",
        model_name="BAAI/bge-small-en-v1.5"
    )
    await ingestor.setup()
    
    for doc in docs_to_ingest:
        print(f"Ingesting {doc.metadata.filename}...")
        await ingestor.process_document(doc)
    
    print("Ingestion complete.")
    return ingestor

if 'docs' in locals():
    ingestor = await run_ingestion(docs)

## 3. Hybrid Reranking & Retrieval

Perform a secure search followed by Cross-Encoder reranking for maximum precision.

In [ ]:
from reranker_pipeline.hybrid_reranker import HybridReranker

async def demo_search(query):
    # Standard search via ingestor
    search_results = await ingestor.secure_search(query=query, limit=10)
    
    # Initialize Reranker
    reranker = HybridReranker(
        model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"
    )
    
    # Rerank the candidates
    final_results = reranker.rerank(query, search_results)
    
    print(f"\nTop results for: '{query}'")
    for i, res in enumerate(final_results[:3]):
        print(f"\n[{i+1}] Score: {res.metadata.get('rerank_score', 0):.4f}")
        print(f"Source: {res.metadata.get('filename')} (Page {res.metadata.get('page_index')})")
        print(f"Excerpt: {res.content[:200]}...")
        
    return final_results

if 'ingestor' in locals():
    final_hits = await demo_search("What are the growth estimates?")

In [ ]:
# Verify integrity of the top result
if 'final_hits' in locals() and final_hits:
    top_doc = final_hits[0]
    is_valid = await ingestor.verify_integrity(
        top_doc.metadata.get('filename'), 
        top_doc.metadata.get('page_index')
    )
    print(f"Integrity of top result ({top_doc.metadata.get('filename')}): {'✓ VERIFIED' if is_valid else '✗ CORRUPTED'}")